# Aspen HYSYS Interface Tutorial

This notebook is meant to teach the practical shape of a Python-to-HYSYS COM
workflow, not just show a finished helper function.

The notebook walks through a realistic sequence:

1. choose a runtime mode
2. connect to the active HYSYS case or use a mock case
3. discover stream and unit-operation names safely
4. map notebook parameters to HYSYS object names
5. inspect values before writing anything
6. apply structured flowsheet updates
7. compare baseline and tuned cases
8. inspect energy-stream usage and utility cost
9. retarget the same pattern to your own case


## Before You Start

Use `mock` mode if any of the following is true:

- you are reading the notebook on GitHub
- you do not have Aspen HYSYS installed
- you are not on Windows
- you want to understand the workflow before touching a real case

Switch to `active_hysys` only when all of the following are true:

- Aspen HYSYS is already open
- the target simulation is the active document in HYSYS
- Python can import `win32com.client`
- you are ready to verify stream and operation names against the live case

The notebook defaults to `mock` so every section can run end to end.


In [ ]:
import sys
from pathlib import Path

import pandas as pd

NOTEBOOK_DIR = Path.cwd().resolve()
DEMO_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
if str(DEMO_DIR) not in sys.path:
    sys.path.insert(0, str(DEMO_DIR))

from hysys_demo import (
    DEFAULT_HDA_SAMPLE,
    HdaFlowsheetMap,
    UTILITY_COST_USD_PER_KJ,
    apply_hda_demo_sample,
    build_mock_context,
    calculate_utility_cost_per_hour,
    collect_energy_table,
    connect_to_active_case,
)

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 180)


## 1. Select the Runtime Mode

The helper package supports two modes:

- `mock`: an in-memory stand-in for the subset of the COM interface used in
  this tutorial
- `active_hysys`: a real COM connection to `HYSYS.Application`

For a real case, the connection line is the important one:

`win32com.client.GetObject(None, "HYSYS.Application")`


In [ ]:
RUNTIME_MODE = 'mock'  # change to 'active_hysys' on Windows with Aspen HYSYS open

if RUNTIME_MODE == 'active_hysys':
    ctx = connect_to_active_case()
elif RUNTIME_MODE == 'mock':
    ctx = build_mock_context()
else:
    raise ValueError("RUNTIME_MODE must be 'mock' or 'active_hysys'")

print('Runtime mode:', RUNTIME_MODE)
print('Case title:', ctx.case.Title)


## 2. Learn How to Discover Names in a Real Case

One of the first practical hurdles in HYSYS automation is simply finding the
correct stream and operation names. Before writing any automation logic,
inspect the case and make a short list of the exact names you want to use.

This section gives a safe discovery pattern you can reuse on a live case.


In [ ]:
def _collection_names(collection):
    names = getattr(collection, 'Names', None)
    if names is not None:
        return list(names)
    mapping = getattr(collection, '_mapping', None)
    if mapping is not None:
        return list(mapping.keys())
    try:
        return [getattr(item, 'Name', str(item)) for item in collection]
    except Exception:
        return []


material_stream_names = _collection_names(ctx.flowsheet.MaterialStreams)
energy_stream_names = _collection_names(ctx.flowsheet.EnergyStreams)
operation_names = _collection_names(getattr(ctx.flowsheet, 'Operations', None))

discovery_df = pd.DataFrame(
    {
        'collection': ['MaterialStreams', 'EnergyStreams', 'Operations'],
        'count': [len(material_stream_names), len(energy_stream_names), len(operation_names)],
        'first_five_names': [
            material_stream_names[:5],
            energy_stream_names[:5],
            operation_names[:5],
        ],
    }
)
discovery_df


In a real HYSYS session, this discovery step is where you confirm the exact
names you will later place into your mapping object. If a name is wrong here,
every downstream automation step becomes brittle.


## 3. Understand the Case Map

The notebook should not hard-code stream and unit names all over the place.
Instead, it should centralize them in one mapping object. That gives you one
place to edit when you switch to another flowsheet or another naming scheme.


In [ ]:
mapping = HdaFlowsheetMap()
mapping_df = pd.DataFrame(
    [{'field': key, 'hysys_name': value} for key, value in vars(mapping).items()]
)
mapping_df


## 4. See How Notebook Parameters Map to COM Writes

The helper function `apply_hda_demo_sample()` turns a compact sample
dictionary into many individual COM writes. This table makes that expansion
explicit.


In [ ]:
write_plan_df = pd.DataFrame(
    [
        {'sample_key': 'H2_flow_kmol_h', 'target_object': mapping.hydrogen_stream, 'target_property': 'MaterialStreams.Item(...).MolarFlow.Value'},
        {'sample_key': 'feed_pressure_kpa', 'target_object': mapping.feed_stream, 'target_property': 'MaterialStreams.Item(...).Pressure.Value'},
        {'sample_key': 'feed_temperature_c', 'target_object': mapping.feed_stream, 'target_property': 'MaterialStreams.Item(...).Temperature.Value'},
        {'sample_key': 'operation_mode', 'target_object': mapping.reactor_outlet_stream, 'target_property': 'MaterialStreams.Item(...).Temperature.Value'},
        {'sample_key': 'reactor_volume_m3', 'target_object': mapping.reactor_name, 'target_property': 'Operations.Item(...).TotalVolumeValue'},
        {'sample_key': 'purge_split_fraction', 'target_object': mapping.purge_splitter, 'target_property': 'Operations.Item(...).SplitsValue'},
        {'sample_key': 'recycle_split_fraction', 'target_object': mapping.recycle_splitter, 'target_property': 'Operations.Item(...).SplitsValue'},
        {'sample_key': 'benzene_column_trays', 'target_object': mapping.benzene_column, 'target_property': 'ColumnFlowsheet TraySection.NumberOfTrays'},
        {'sample_key': 'toluene_column_trays', 'target_object': mapping.toluene_column, 'target_property': 'ColumnFlowsheet TraySection.NumberOfTrays'},
        {'sample_key': 'benzene_feed_stage_ratio', 'target_object': mapping.benzene_column, 'target_property': 'TraySection.SpecifyFeedLocation(...)'},
        {'sample_key': 'toluene_feed_stage_ratio', 'target_object': mapping.toluene_column, 'target_property': 'TraySection.SpecifyFeedLocation(...)'},
    ]
)
write_plan_df


## 5. Inspect Objects Before Writing Anything

A good HYSYS automation workflow begins with small read-only checks. That
makes it easy to catch a wrong stream name or a wrong unit before you start
overwriting values.


In [ ]:
def snapshot_streams(stream_names):
    rows = []
    for name in stream_names:
        stream = ctx.flowsheet.MaterialStreams.Item(name)
        rows.append(
            {
                'stream': name,
                'temperature_c': getattr(stream.Temperature, 'Value', None),
                'pressure_kpa': getattr(stream.Pressure, 'Value', None),
                'molar_flow_kmol_s': getattr(stream.MolarFlow, 'Value', None),
            }
        )
    return pd.DataFrame(rows)


inspection_df = snapshot_streams(
    [
        mapping.hydrogen_stream,
        mapping.feed_stream,
        mapping.reactor_outlet_stream,
        mapping.condenser_stream,
        mapping.benzene_column_stream,
        mapping.toluene_column_stream,
    ]
)
inspection_df


## 6. Minimal COM Sanity Check Pattern

When you move from mock mode to a real case, keep the first write trivial.
The point is to confirm that your Python object really points to the intended
HYSYS object.

The pattern below is deliberately conservative:

- read the current value
- if needed, write the same value back first
- only then start making real changes through your helper function


In [ ]:
feed_stream = ctx.flowsheet.MaterialStreams.Item(mapping.feed_stream)
current_feed_pressure = getattr(feed_stream.Pressure, 'Value', None)

sanity_check_df = pd.DataFrame(
    [
        {
            'object': mapping.feed_stream,
            'property': 'Pressure.Value',
            'current_value': current_feed_pressure,
            'safe_first_action': 'read it, then optionally write the same value back',
        }
    ]
)
sanity_check_df


## 7. Review the Baseline Input Sample

The sample dictionary is the notebook-facing representation of the flowsheet
changes we want to apply. The helper package translates those high-level
fields into the COM writes listed above.


In [ ]:
baseline_sample_df = pd.DataFrame(
    [{'parameter': key, 'value': value} for key, value in DEFAULT_HDA_SAMPLE.items()]
)
baseline_sample_df


## 8. Apply the Baseline Sample

`apply_hda_demo_sample()` performs the structured writes. In this demo it:

- sets the hydrogen-feed flow
- updates the reactor-feed temperature and pressure
- links the secondary feed-stream pressure when needed
- updates condenser and column pressures
- updates reactor volume
- updates purge and recycle splitter fractions
- updates column tray counts and feed-stage ratios


In [ ]:
baseline_applied = apply_hda_demo_sample(DEFAULT_HDA_SAMPLE, mapping=mapping, context=ctx)

baseline_snapshot_df = snapshot_streams(
    [
        mapping.hydrogen_stream,
        mapping.feed_stream,
        mapping.reactor_outlet_stream,
        mapping.condenser_stream,
    ]
)
baseline_snapshot_df


## 9. Create and Apply a Tuned Case

Once the baseline mapping works, the normal next step is to clone the sample,
adjust a few values, and run again. This is the point where you usually start
embedding the workflow into optimization, sensitivity analysis, or DOE loops.


In [ ]:
tuned_sample = dict(DEFAULT_HDA_SAMPLE)
tuned_sample.update(
    {
        'operation_mode': 'isothermal',
        'feed_pressure_kpa': 3450.0,
        'feed_temperature_c': 640.0,
        'recycle_split_fraction': 0.86,
        'purge_split_fraction': 0.08,
        'benzene_column_trays': 48,
        'toluene_column_trays': 30,
    }
)

tuned_sample_df = pd.DataFrame(
    [{'parameter': key, 'value': value} for key, value in tuned_sample.items()]
)
tuned_sample_df


In [ ]:
def run_case(label, sample):
    applied = apply_hda_demo_sample(sample, mapping=mapping, context=ctx)
    energy_df = collect_energy_table(context=ctx)
    return {
        'label': label,
        'operation_mode': applied['operation_mode'],
        'feed_pressure_kpa': applied['feed_pressure_kpa'],
        'feed_temperature_c': applied['feed_temperature_c'],
        'recycle_split_fraction': applied['recycle_split_fraction'],
        'purge_split_fraction': applied['purge_split_fraction'],
        'benzene_column_trays': applied['benzene_column_trays'],
        'toluene_column_trays': applied['toluene_column_trays'],
        'energy_stream_count': len(energy_df),
        'utility_cost_usd_per_h': calculate_utility_cost_per_hour(context=ctx),
    }


comparison_df = pd.DataFrame(
    [
        run_case('baseline', DEFAULT_HDA_SAMPLE),
        run_case('tuned', tuned_sample),
    ]
)
comparison_df


## 10. Inspect the Energy Streams

After applying a case, the next useful check is usually the energy-stream
table. This confirms which duties are present and how the helper grouped them
into utility categories.


In [ ]:
energy_df = collect_energy_table(context=ctx)
energy_df


In [ ]:
utility_summary_df = (
    energy_df.groupby('utility_category', dropna=False)[['heat_flow_kw', 'heat_flow_kj_h']]
    .sum()
    .reset_index()
    .sort_values('utility_category')
)
utility_summary_df['estimated_cost_usd_per_h'] = utility_summary_df.apply(
    lambda row: row['heat_flow_kj_h'] * UTILITY_COST_USD_PER_KJ[row['utility_category']],
    axis=1,
)
utility_summary_df


## 11. Suggested Workflow on a Real HYSYS Case

A practical order of operations is:

1. open HYSYS and make sure the target case is the active document
2. run the notebook in `mock` mode once so you understand the structure
3. switch to `active_hysys`
4. use the discovery cells to list real stream and operation names
5. edit `HdaFlowsheetMap` until the read-only inspection cells point to the
   correct objects
6. perform a minimal sanity check on one object
7. only then use `apply_hda_demo_sample()` or your own helper function
8. keep all real COM write logic inside helper modules, not scattered through
   the notebook


## 12. Common Failure Modes and How to Debug Them

If the notebook fails on a real case, check these in order:

1. `win32com.client` import fails
   - Python environment issue, not a HYSYS issue

2. `GetObject(None, "HYSYS.Application")` fails
   - HYSYS is not open, or COM registration is unavailable

3. `ActiveDocument` is wrong or empty
   - the wrong case is active in the HYSYS UI

4. `MaterialStreams.Item(...)` or `Operations.Item(...)` fails
   - name mismatch between notebook mapping and the actual case

5. properties exist but values look wrong
   - you are connected to the wrong object, unit, or case version

6. writes succeed but downstream behavior is inconsistent
   - your helper logic is changing dependent objects in the wrong order

The first debugging target is almost always the mapping, not the optimizer
or outer workflow.


## 13. Where to Extend Next

Once this notebook is stable on your case, the next useful upgrades are:

- parameter sweeps across a table of candidate samples
- CSV logging of each evaluated case
- structured exception handling around failed HYSYS solves
- notebook-free batch runners that call the same helper package
- optimization loops that treat `apply_hda_demo_sample()` as the write step
  and your indicator functions as the read step
